# Math Prerequisites → Linear Algebra | Chapter 3: Singular Value Decomposition (SVD)

> **Why this chapter exists:** Every modern PCA implementation uses SVD — not eigendecomposition. `sklearn.decomposition.PCA` calls `np.linalg.svd` internally. Understanding SVD is therefore not optional — it is how PCA actually works in practice. SVD generalizes eigendecomposition to *any* matrix (not just square symmetric ones), and the connection between SVD and PCA is exact and beautiful.

---

## 1. Why SVD? The Limitation of Eigendecomposition

Eigendecomposition $A = Q\Lambda Q^{-1}$ has restrictions:
1. **Square matrices only** — $A$ must be $n \times n$.
2. **Full rank required** — $A$ must have $n$ linearly independent eigenvectors.
3. **May produce complex eigenvalues** — for non-symmetric matrices.

For PCA, the data matrix $X \in \mathbb{R}^{N \times d}$ is almost never square ($N \neq d$). We want to decompose $X$ directly — not first compute $\Sigma = X^T X / (N-1)$.

**SVD solves all these problems:** it decomposes **any** $m \times n$ matrix into a product of three structured matrices, with real non-negative values.

---

## 2. The SVD Definition

Every matrix $A \in \mathbb{R}^{m \times n}$ (regardless of shape or rank) can be factored as:

$$\boxed{A = U \Sigma V^T}$$

where:
- $U \in \mathbb{R}^{m \times m}$ — **Left singular vectors** (orthonormal columns): $U^T U = I$
- $\Sigma \in \mathbb{R}^{m \times n}$ — **Singular values** (diagonal, non-negative, decreasing): $\sigma_1 \geq \sigma_2 \geq \cdots \geq 0$
- $V \in \mathbb{R}^{n \times n}$ — **Right singular vectors** (orthonormal columns): $V^T V = I$

The singular values live on the diagonal of $\Sigma$: $\Sigma_{ii} = \sigma_i$, all other entries zero.

### **Thin (Economy) SVD vs Full SVD**

For $m > n$ (more rows than columns — tall matrix), most of $U$ and $\Sigma$ are zero-padded:
- **Full SVD:** $U$ is $m \times m$, $\Sigma$ is $m \times n$ — includes zero rows
- **Thin SVD (economy):** $U$ is $m \times r$, $\Sigma$ is $r \times r$, $V$ is $n \times r$, where $r = \min(m,n)$

In code: `np.linalg.svd(A, full_matrices=False)` gives the thin SVD. **Always use thin SVD for PCA** — it's equivalent but much smaller when $N \gg d$.

---

## 2. Geometric Interpretation: Rotation → Scale → Rotation

The product $A = U\Sigma V^T$ decomposes any linear transformation into three interpretable steps:

$$\vec{y} = A\vec{x} = U \cdot \Sigma \cdot V^T \cdot \vec{x}$$

Apply right-to-left:

| Step | Operation | Matrix | Effect |
| :---: | :--- | :---: | :--- |
| 1 | $\vec{x}' = V^T\vec{x}$ | $V^T$ | **Rotate** input to align with right singular vectors |
| 2 | $\vec{x}'' = \Sigma\vec{x}'$ | $\Sigma$ | **Scale** each dimension by $\sigma_i$ (stretch/shrink) |
| 3 | $\vec{y} = U\vec{x}''$ | $U$ | **Rotate** output to output space |

So every matrix transformation = **two rotations + one axis-aligned scaling**. The singular values $\sigma_i$ tell you how much the matrix stretches space along each direction.

**Analogy:** Imagine an ellipse. Any linear transformation maps the unit ball to an ellipsoid. The **axes of the ellipsoid** are the left singular vectors $U$. The **radii of the ellipsoid** are the singular values $\sigma_i$. The **input directions** that map to those axes are the right singular vectors $V$.

---

## 3. Singular Values vs Eigenvalues: The Connection

### **Where Singular Values Come From**

Multiply $A = U\Sigma V^T$ by its transpose:

$$A^T A = (U\Sigma V^T)^T (U\Sigma V^T) = V \Sigma^T U^T U \Sigma V^T = V (\Sigma^T \Sigma) V^T$$

This is the **eigendecomposition of $A^T A$**! Therefore:
- **Right singular vectors** $V$ = eigenvectors of $A^T A$
- **Squared singular values** $\sigma_i^2$ = eigenvalues of $A^T A$
$$\boxed{\sigma_i = \sqrt{\lambda_i(A^T A)}}$$

Similarly, $AA^T = U(\Sigma\Sigma^T)U^T$, so:
- **Left singular vectors** $U$ = eigenvectors of $AA^T$

### **Connection to PCA Covariance Matrix**
The PCA covariance matrix $\Sigma = X^T X / (N-1)$. So:
$$\underbrace{X^T X}_{\text{unnorm. covariance}} = V \underbrace{(\Sigma_s^T \Sigma_s)}_{\sigma_i^2 \text{ on diagonal}} V^T$$

Dividing by $(N-1)$:
$$\text{Cov}(X) = V \cdot \frac{\text{diag}(\sigma_i^2)}{N-1} \cdot V^T$$

Therefore:
- **Principal components** = right singular vectors $V$ of $X$
- **Variance of PC $k$** = $\sigma_k^2 / (N-1)$
- **PC scores** = $U \cdot \Sigma_s$ (equivalently: $Z = X V$)

**This is why SVD of $X$ directly gives PCA** — no need to compute the covariance matrix.

---

## 4. Truncated SVD: The Best Low-Rank Approximation

### **The Rank-1 Outer Product Expansion**
Just like eigendecomposition, SVD can be written as a sum of rank-1 matrices:
$$A = U\Sigma V^T = \sum_{i=1}^{r} \sigma_i \vec{u}_i \vec{v}_i^T$$

Each term $\sigma_i \vec{u}_i \vec{v}_i^T$ is a rank-1 matrix (outer product of two vectors).

### **Eckart-Young Theorem**

> **Theorem (Eckart & Young, 1936):** The best rank-$k$ approximation to $A$ in the Frobenius norm (or spectral norm) is the **truncated SVD**:
> $$A_k = \sum_{i=1}^{k} \sigma_i \vec{u}_i \vec{v}_i^T = U_k \Sigma_k V_k^T$$
> where $U_k, V_k$ are the first $k$ columns of $U, V$ and $\Sigma_k = \text{diag}(\sigma_1,...,\sigma_k)$.
> The approximation error:
> $$\|A - A_k\|_F^2 = \sum_{i=k+1}^r \sigma_i^2$$

**PCA connection:** In PCA, the reconstruction of centered data using $k$ PCs:
$$\hat{X} = Z V_k^T = (X V_k) V_k^T = X V_k V_k^T$$
This is exactly the rank-$k$ truncated SVD of $X$. **PCA finds the best rank-$k$ linear approximation to the data.**

### **Explained Variance via Singular Values**
$$\text{Explained variance ratio for PC } k = \frac{\sigma_k^2}{\sum_{i=1}^r \sigma_i^2}$$

---

## 5. Why SVD Over Eigendecomposition in Practice?

Both give the same result. SVD is preferred for three reasons:

### **1. Numerical Stability**
Computing $\Sigma = X^T X$ and then eigendecomposing squares the condition number of the problem. If $X$ has a condition number $\kappa$, then $X^T X$ has condition number $\kappa^2$ — making the eigensolution twice as numerically sensitive. SVD directly operates on $X$, avoiding this squaring.

**Concrete example:** If $X$ has singular values $[1000, 0.001]$, then $X^T X$ has eigenvalues $[10^6, 10^{-6}]$ — a ratio of $10^{12}$, which is at the floating-point precision limit for 64-bit numbers.

### **2. Works for Any Shape**
SVD works on $X$ directly whether $N > d$, $N < d$, or any rank. Eigendecomposition of $\Sigma$ requires $\Sigma$ to be full-rank — which fails when $N < d$.

### **3. More Information**
SVD gives you both $U$ (left singular vectors = PC scores normalized) and $V$ (right singular vectors = PC directions) simultaneously, without a separate projection step.

---

## 6. ✍️ Worked Example: SVD of a 3×2 Matrix

Let $A = \begin{bmatrix} 1 & 1 \\ 2 & 1 \\ 0 & 1 \end{bmatrix} \in \mathbb{R}^{3 \times 2}$.

**Step 1:** Compute $A^T A \in \mathbb{R}^{2 \times 2}$:
$$A^T A = \begin{bmatrix} 1 & 2 & 0 \\ 1 & 1 & 1 \end{bmatrix} \begin{bmatrix} 1 & 1 \\ 2 & 1 \\ 0 & 1 \end{bmatrix} = \begin{bmatrix} 5 & 3 \\ 3 & 3 \end{bmatrix}$$

**Step 2:** Eigendecompose $A^T A$:
$$\det(A^T A - \lambda I) = (5-\lambda)(3-\lambda) - 9 = \lambda^2 - 8\lambda + 6 = 0$$
$$\lambda_1 = 4 + \sqrt{10} \approx 7.162, \quad \lambda_2 = 4 - \sqrt{10} \approx 0.838$$

**Step 3:** Singular values: $\sigma_1 = \sqrt{\lambda_1} \approx 2.677$, $\sigma_2 = \sqrt{\lambda_2} \approx 0.915$

**Step 4:** $V$ = eigenvectors of $A^T A$ (right singular vectors).

**Step 5:** $U$ = compute $\vec{u}_i = A\vec{v}_i / \sigma_i$ (left singular vectors).

**Step 6:** Verify: $A \approx \sigma_1 \vec{u}_1 \vec{v}_1^T + \sigma_2 \vec{u}_2 \vec{v}_2^T$ (sum of 2 rank-1 matrices).

---

## 7. SVD ↔ PCA: Complete Lookup Table

| PCA Concept | SVD of centered $X$ |
| :--- | :--- |
| Principal component directions | Right singular vectors: columns of $V$ |
| PC scores | $Z = U\Sigma_s$ (or equivalently $Z = XV$) |
| Variance of PC $k$ | $\sigma_k^2 / (N-1)$ |
| Explained variance ratio | $\sigma_k^2 / \sum_i \sigma_i^2$ |
| Reconstruction of data | $\hat{X} = U_k \Sigma_k V_k^T = Z_k V_k^T$ |
| Reconstruction error | $\sum_{i=k+1}^r \sigma_i^2 / (N-1)$ |
| Number of non-zero PCs | $r = \text{rank}(X) \leq \min(N, d)$ |

---

## 8. Implementation Playground

In [ ]:
# ─── CELL 1: SVD Manual Verification + PCA Connection ─────────────────────────
import numpy as np

# Small worked example
A = np.array([[1., 1.], [2., 1.], [0., 1.]])  # 3x2 matrix

# Full SVD
U, s, Vt = np.linalg.svd(A, full_matrices=False)  # thin SVD
Sigma = np.diag(s)

print(f"A shape: {A.shape}")
print(f"U (left singular vectors): shape {U.shape}")
print(U.round(4))
print(f"\nSingular values: {s.round(4)}")
print(f"V^T (right singular vectors transposed): shape {Vt.shape}")
print(Vt.round(4))

# Verify: A = U @ Sigma @ V^T
A_reconstructed = U @ Sigma @ Vt
print(f"\nA = U Σ Vᵀ reconstructed perfectly? {np.allclose(A, A_reconstructed)}")

# Rank-1 expansion: A = σ₁u₁v₁ᵀ + σ₂u₂v₂ᵀ
A_rank1 = s[0] * np.outer(U[:,0], Vt[0,:])
A_rank2 = s[0] * np.outer(U[:,0], Vt[0,:]) + s[1] * np.outer(U[:,1], Vt[1,:])
print(f"\nRank-1 approximation error: {np.linalg.norm(A - A_rank1, 'fro'):.4f}")
print(f"Eckart-Young theorem: should equal σ₂ = {s[1]:.4f}")
print(f"Rank-2 approximation (full): error = {np.linalg.norm(A - A_rank2, 'fro'):.6f}")

# SVD vs eigendecomposition of A^T A
AtA = A.T @ A
evals, evecs = np.linalg.eigh(AtA)
evals_sorted = evals[::-1]; evecs_sorted = evecs[:, ::-1]
print(f"\nSingular values² from SVD: {(s**2).round(4)}")
print(f"Eigenvalues of AᵀA:         {evals_sorted.round(4)}")
print(f"Match? {np.allclose(s**2, evals_sorted)}")

In [ ]:
# ─── CELL 2: SVD vs Covariance Eigendecomposition — Prove They Give Same PCA ──
import numpy as np
from sklearn.decomposition import PCA

np.random.seed(42)
N, d = 200, 5
X_raw = np.random.randn(N, d)
# Add correlations
X_raw[:, 1] += 0.8 * X_raw[:, 0]
X_raw[:, 3] -= 0.6 * X_raw[:, 2]

# Center the data (CRITICAL: SVD-based PCA requires centered data)
X = X_raw - X_raw.mean(axis=0)

# ── Method 1: Via SVD of X ────────────────────────────────────────────────────
U, s, Vt = np.linalg.svd(X, full_matrices=False)
V_svd = Vt.T                      # PCs = columns of V
Z_svd = U * s                      # Scores = U @ diag(s)
variances_svd = s**2 / (N - 1)    # Variance of each PC

# ── Method 2: Via Covariance Eigendecomposition ───────────────────────────────
cov = X.T @ X / (N - 1)           # d x d covariance matrix
eigenvalues, V_eig = np.linalg.eigh(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
V_eig = V_eig[:, idx]             # PCs = columns
Z_eig = X @ V_eig                 # Scores

# ── Method 3: sklearn PCA ─────────────────────────────────────────────────────
pca = PCA(n_components=d)
Z_sk = pca.fit_transform(X)

# Compare
print("PC directions (columns — may have sign flip, both are valid):")
print("SVD V:  ", V_svd[:, 0].round(4))
print("Eig V:  ", V_eig[:, 0].round(4))
print("Sklearn:", pca.components_[0].round(4))

print("\nVariances per PC:")
print("SVD:    ", variances_svd.round(4))
print("Eig:    ", eigenvalues.round(4))
print("Sklearn:", pca.explained_variance_.round(4))

print("\nAll three methods give same variances?",
      np.allclose(variances_svd, eigenvalues, atol=1e-6) and
      np.allclose(eigenvalues, pca.explained_variance_, atol=1e-6))

In [ ]:
# ─── CELL 3: Image Compression via Truncated SVD (Eckart-Young) ───────────────
import numpy as np
import matplotlib.pyplot as plt

# Create a structured synthetic image (or use a real one)
np.random.seed(0)
size = 128
x_grid = np.linspace(0, 4 * np.pi, size)
y_grid = np.linspace(0, 4 * np.pi, size)
X_img, Y_img = np.meshgrid(x_grid, y_grid)
M = np.sin(X_img) * np.cos(Y_img) + 0.3 * np.random.randn(size, size)

# Full SVD
U, s, Vt = np.linalg.svd(M, full_matrices=False)

# Truncated reconstructions
ks = [1, 3, 10, 30, 64]
total_params_full = size * size

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

axes[0].imshow(M, cmap='RdBu_r'); axes[0].set_title('Original (128×128)', fontsize=11)
axes[0].axis('off')

for i, k in enumerate(ks):
    M_k = (U[:, :k] * s[:k]) @ Vt[:k, :]   # Rank-k approximation
    evr = (s[:k]**2).sum() / (s**2).sum() * 100
    params = k * (1 + size + size)  # k singular values + k left + k right vectors
    compression = total_params_full / params
    axes[i+1].imshow(M_k, cmap='RdBu_r')
    axes[i+1].set_title(f'k={k} | {evr:.1f}% var | {compression:.0f}x compression', fontsize=10)
    axes[i+1].axis('off')

plt.suptitle('Truncated SVD = Best Rank-k Approximation (Eckart-Young Theorem)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# Plot singular value spectrum
fig2, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogy(s, 'o-', markersize=3, color='steelblue')
ax1.set_xlabel('Rank k'); ax1.set_ylabel('Singular value (log scale)')
ax1.set_title('Singular Value Spectrum'); ax1.grid(alpha=0.3)

cumvar = np.cumsum(s**2) / (s**2).sum() * 100
ax2.plot(cumvar, color='darkorange', lw=2)
ax2.axhline(95, color='red', ls='--', label='95% threshold')
ax2.set_xlabel('Number of singular values k'); ax2.set_ylabel('Cumulative variance (%)')
ax2.set_title('Cumulative Explained Variance'); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()